In [49]:
import pandas as pd
import numpy as np
import glob
import os

In [50]:
# Merger data
folder_path=r'C:\Users\Admin\Memories\Desktop\DA\Project3\2024'
all_files = glob.glob(os.path.join(folder_path,"*.csv"))
df=pd.concat([pd.read_csv(file) for file in all_files], ignore_index=True)
print("Dataset Shape:", df.shape)

Dataset Shape: (5860568, 13)


In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5860568 entries, 0 to 5860567
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 581.3+ MB


In [52]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,C1D650626C8C899A,electric_bike,1/12/2024 15:30,1/12/2024 15:37,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.903267,-87.634737,41.889177,-87.638506,member
1,EECD38BDB25BFCB0,electric_bike,1/8/2024 15:45,1/8/2024 15:52,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902937,-87.634440,41.889177,-87.638506,member
2,F4A9CE78061F17F7,electric_bike,1/27/2024 12:27,1/27/2024 12:35,Wells St & Elm St,KA1504000135,Kingsbury St & Kinzie St,KA1503000043,41.902951,-87.634470,41.889177,-87.638506,member
3,0A0D9E15EE50B171,classic_bike,1/29/2024 16:26,1/29/2024 16:56,Wells St & Randolph St,TA1305000030,Larrabee St & Webster Ave,13193,41.884295,-87.633963,41.921822,-87.644140,member
4,33FFC9805E3EFF9A,classic_bike,1/31/2024 5:43,1/31/2024 6:09,Lincoln Ave & Waveland Ave,13253,Kingsbury St & Kinzie St,KA1503000043,41.948797,-87.675278,41.889177,-87.638506,member


In [53]:
df.isnull().sum()

ride_id                     0
rideable_type               0
started_at                  0
ended_at                    0
start_station_name    1073951
start_station_id      1073951
end_station_name      1104653
end_station_id        1104653
start_lat                   0
start_lng                   0
end_lat                  7232
end_lng                  7232
member_casual               0
dtype: int64

**Observations:**
The output reveals missing data primarily concentrated in the station information and end coordinates:
* **`start_station_name` & `start_station_id`:** 1,073,951 missing values.
* **`end_station_name` & `end_station_id`:** 1,104,653 missing values.
* **`end_lat` & `end_lng`:** 7,232 missing values.
* Core metrics such as `ride_id`, timestamps, and `member_casual` are fully populated (0 missing values).

In [54]:
df.duplicated().sum()

np.int64(0)

In [55]:
initial_rows = len(df)
cleaning_log = []

In [56]:
df[['start_station_name',
    'start_station_id',
    'end_station_name',
    'end_station_id']].isnull().sum()

start_station_name    1073951
start_station_id      1073951
end_station_name      1104653
end_station_id        1104653
dtype: int64

# Handling Missing Station Data
To account for dockless or free-floating bike usage, fill the missing values in the station name and ID columns with 'On Street - Dockless' and 'SYS_DOCKLESS' respectively, rather than dropping the records.

In [57]:
df['start_station_name'] = df['start_station_name'].fillna('On Street - Dockless')
df['start_station_id'] = df['start_station_id'].fillna('SYS_DOCKLESS')

df['end_station_name'] = df['end_station_name'].fillna('On Street - Dockless')
df['end_station_id'] = df['end_station_id'].fillna('SYS_DOCKLESS')

In [59]:
before = len(df)

In [60]:
df = df.dropna(subset=['end_lat', 'end_lng'])

In [61]:
after = len(df)
cleaning_log.append(['Removed missing coordinates', before - after])

In [63]:
df['started_at'] = pd.to_datetime(df['started_at'],format='mixed')
df['ended_at'] = pd.to_datetime(df['ended_at'], format='mixed')

# Checking Datetime Data Types
Verify the data types of the `started_at` and `ended_at` columns to ensure they are correctly formatted as datetime objects.

In [64]:
df[['started_at','ended_at']].dtypes

started_at    datetime64[ns]
ended_at      datetime64[ns]
dtype: object

In [ ]:
**Observations:**
The output confirms that both `started_at` and `ended_at` are of type `datetime64[ns]`. This indicates they are correctly parsed as datetime objects and are ready for time-based operations, such as calculating ride durations.

In [68]:
df['ride_length_minutes'] = (
    df['ended_at'] - df['started_at']
).dt.total_seconds() / 60

# Ride Length Statistics
Generate descriptive statistics for the `ride_length_minutes` column to understand the distribution of ride durations and identify any potential anomalies.

In [69]:
df['ride_length_minutes'].describe()

count    5.853336e+06
mean     1.548389e+01
std      3.287441e+01
min     -2.748317e+03
25%      5.541067e+00
50%      9.707100e+00
75%      1.720049e+01
max      1.509367e+03
Name: ride_length_minutes, dtype: float64

**Observations:**
The summary statistics reveal critical insights and data quality issues:
* **Negative Values:** The minimum value is approximately -2748 minutes. Negative ride durations are logically impossible and indicate system logging errors or incorrect timestamps.
* **Extreme Outliers:** The maximum duration is over 1509 minutes (approx. 25 hours), which likely points to unclosed trips, abandoned bikes, or system testing.
* **Central Tendency:** The median (50%) ride time is 9.7 minutes, and 75% of all rides are under 17.2 minutes, confirming that the vast majority of trips are short-distance rides.

# Removing Invalid Ride Durations
Filter the dataset to exclude records where `ride_length_minutes` is zero or negative, as these are logically invalid and likely represent system logging errors. The number of removed records is tracked in `cleaning_log`.

In [70]:
before = len(df)

df = df[
    df['ride_length_minutes'] > 0
]

after = len(df)

cleaning_log.append([
    'Removed negative durations',
    before - after
])

# Removing Short Rides
Filter out trips that are less than 1 minute long. These extremely short durations typically represent false starts, immediate cancellations, or system testing rather than actual rides. The count of removed records is recorded in the `cleaning_log`.

In [72]:
before = len(df)

df = df[
    df['ride_length_minutes'] >= 1
]

after = len(df)

cleaning_log.append([
    'Removed rides under 1 minute',
    before - after
])

# Analyzing Upper Percentiles
Re-evaluate the descriptive statistics of `ride_length_minutes`, specifically including the 95th and 99th percentiles, to understand the distribution of the upper tail and identify extreme outliers after the initial cleaning steps.

In [73]:
df['ride_length_minutes'].describe(
    percentiles=[0.95,0.99])

count    5.723636e+06
mean     1.582775e+01
std      3.313953e+01
min      1.000000e+00
50%      9.939417e+00
95%      4.272353e+01
99%      9.594593e+01
max      1.509367e+03
Name: ride_length_minutes, dtype: float64

**Observations:**
* **Sanity Check:** The minimum ride length is now 1 minute, confirming the success of the previous filtering steps.
* **Upper Percentiles:** 95% of all rides are completed within ~42.7 minutes, and 99% are completed within ~96 minutes.
* **Extreme Outliers:** Despite 99% of the data falling under 1.6 hours, the maximum value remains at 1,509 minutes (approx. 25 hours). This massive right skew indicates that the top 1% contains extreme outliers (such as stolen, lost, or un-docked bikes) that may require capping or removal to prevent skewed aggregations.

# Inspecting Extreme Outliers
Examine the top 10 longest rides in the dataset to investigate the characteristics of the extreme outliers identified in the upper percentiles.

In [74]:
df.nlargest(
    10,
    'ride_length_minutes'
)[[
    'ride_id',
    'ride_length_minutes',
    'member_casual'
]]

,ride_id,ride_length_minutes,member_casual
423488,7A5CAAC52FAE9E95,1509.366667,casual
948210,3F95397BA3FDE147,1500.516667,casual
77911,395F67B461B54C8D,1500.000000,casual
78011,A8A1271F39C6865C,1500.000000,casual
78029,8DB4F97C5E7B33C7,1500.000000,member
82415,4A59ACAAD442ADA4,1500.000000,member
90486,D1FD72EF84FD52B2,1500.000000,member
90651,1A3C7CC0DDE14A9A,1500.000000,casual
90714,1C77736FC6C6E417,1500.000000,member
90834,E6B258E01E383B6D,1500.000000,casual


**Observations:**
The output reveals a distinct pattern regarding the maximum ride lengths:
* **System Timeout Cap:** Many of the absolute longest rides have an identical duration of exactly 1,500 minutes (25 hours). This strongly suggests a system-enforced timeout, auto-closure mechanism, or hard cap for unreturned bikes rather than actual continuous riding.
* **User Types Affected:** These extreme durations are present across both `casual` riders and `member` riders, indicating an operational or system-level occurrence rather than behavior isolated to a single customer segment.

# User Type Distribution
Calculate the total number of trips taken by each user type (`member` vs. `casual`) to understand the overall composition of the customer base.

In [75]:
df['member_casual'].value_counts()

member_casual
member    3642920
casual    2080716
Name: count, dtype: int64

**Observations:**
The output displays the total volume of rides split by customer segment:
* **Annual Members (`member`):** 3,642,920 trips.
* **Casual Riders (`casual`):** 2,080,716 trips.
* Annual members account for the clear majority of the trips in the dataset, representing approximately 63.6% of the total ride volume compared to 36.4% for casual riders.

# Rideable Type Distribution
Analyze the usage frequency of different vehicle types (`rideable_type`) to understand fleet preferences and utilization across the platform.

In [76]:
df['rideable_type'].value_counts()

rideable_type
electric_bike       2870363
classic_bike        2715689
electric_scooter     137584
Name: count, dtype: int64

**Observations:**
The output details the total volume of trips by vehicle type:
* **Electric Bikes (`electric_bike`):** The most popular choice, accounting for 2,870,363 trips.
* **Classic Bikes (`classic_bike`):** A close second with 2,715,689 trips, indicating a strong continued reliance on traditional bicycles.
* **Electric Scooters (`electric_scooter`):** Make up a significantly smaller portion of the overall usage with 137,584 trips.

# Extracting Temporal Features
Extract the day of the week and the hour of the day from the `started_at` timestamp to enable granular, time-based behavioral analysis.

In [78]:
df['day_of_week'] = df['started_at'].dt.day_name()
df['day_of_week'] = df['started_at'].dt.day_name()
df['hour'] = df['started_at'].dt.hour

# Checking for Duplicate Records
Verify the uniqueness of the `ride_id` column, which serves as the primary key for the dataset, by counting the number of duplicated entries.

In [79]:
df['ride_id'].duplicated().sum()

np.int64(173)

**Observations:**
The output reveals 173 duplicated `ride_id` records. Since `ride_id` should be a unique identifier for each individual trip, these duplicates indicate a minor data quality issue. These duplicate rows will need to be removed to maintain data integrity before proceeding with further analysis.

In [80]:
before = len(df)

df = df.drop_duplicates(
    subset=['ride_id']
)

after = len(df)

cleaning_log.append([
    'Removed duplicate ride_id',
    before - after
])

In [81]:
print(df.shape)

(5723463, 16)


# Final Check for Missing Values
Verify the dataset to ensure that all previously identified missing values have been successfully handled and no nulls remain in the newly engineered columns.

In [82]:
df.isnull().sum()

ride_id                0
rideable_type          0
started_at             0
ended_at               0
start_station_name     0
start_station_id       0
end_station_name       0
end_station_id         0
start_lat              0
start_lng              0
end_lat                0
end_lng                0
member_casual          0
ride_length_minutes    0
day_of_week            0
hour                   0
dtype: int64

**Observations:**
The output confirms that there are exactly 0 missing values across all columns, including the newly added features (`ride_length_minutes`, `day_of_week`, and `hour`). The dataset is now completely populated, clean, and ready for exploratory data analysis or export.

In [83]:
df['ride_id'].duplicated().sum()

np.int64(0)

# Data Cleaning Summary Report
Generate a summary dataframe from the `cleaning_log` to review the sequence of data cleaning steps and quantify the number of records removed at each stage.

In [85]:
cleaning_report = pd.DataFrame(
    cleaning_log,
    columns=[
        'Cleaning Step',
        'Rows Removed'
    ]
)

cleaning_report

,Cleaning Step,Rows Removed
0,Removed missing coordinates,7232
1,Removed negative durations,3132
2,Removed rides under 1 minute,126568
3,Removed rides under 1 minute,0
4,Removed duplicate ride_id,173


**Observations:**
The cleaning report provides a transparent audit trail of the data manipulation process:
* **Primary Data Loss:** The most significant reduction occurred when removing rides under 1 minute (126,568 rows), effectively filtering out false starts or immediate cancellations.
* **Redundant Step:** Step 3 indicates a duplicate execution ("Removed rides under 1 minute" with 0 rows removed), showing that the previous pass successfully cleared all such records.
* **Overall Impact:** Minor cleaning was required for missing coordinates (7,232), negative durations (3,132), and duplicate IDs (173). The total number of removed rows represents a very small fraction of the original multi-million row dataset, preserving the overall data integrity and statistical significance.

# Final Dataset Overview
Generate a high-level summary of the final cleaned dataset, including the total row count, temporal scope, and distributions of key categorical variables (user types and vehicle types).

In [86]:
print("Rows:", len(df))

print("\nDate Range:")
print(df['started_at'].min())
print(df['started_at'].max())

print("\nMembership Types:")
print(df['member_casual'].value_counts())

print("\nBike Types:")
print(df['rideable_type'].value_counts())

Rows: 5723463

Date Range:
2024-01-01 00:00:00
2024-12-31 23:54:37.045000

Membership Types:
member_casual
member    3642843
casual    2080620
Name: count, dtype: int64

Bike Types:
rideable_type
electric_bike       2870276
classic_bike        2715603
electric_scooter     137584
Name: count, dtype: int64


**Observations:**
The output provides a comprehensive snapshot of the fully cleaned dataset ready for analysis:
* **Total Volume:** The dataset contains a robust 5,723,463 valid ride records.
* **Temporal Scope:** The data spans exactly one full calendar year, from January 1, 2024, to December 31, 2024, which is ideal for capturing complete seasonal and annual trends.
* **User Types:** Annual members (3,642,843) continue to represent the majority of total trips compared to casual riders (2,080,620).
* **Vehicle Types:** Electric bikes (2,870,276) and classic bikes (2,715,603) share nearly equal popularity, dominating the fleet usage, while electric scooters (137,584) account for a very small fraction of total rides.

In [87]:
df.to_csv(
    'cyclistic_2024_cleaned.csv',
    index=False
)

print("Export completed.")

Export completed.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv('cyclistic_2024_cleaned.csv')

server = 'DESKTOP-3DJLNHS'
database = 'cyclistic_db'

engine = create_engine(
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes",
    fast_executemany=True  
)

df.to_sql(
    name='Trips', 
    con=engine, 
    if_exists='replace', 
    index=False, 
    chunksize=100000    
)

print("Success")